# Three-layer validation walkthrough

This notebook follows demos `01`, `02` and `03`: the same three validation
layers (structural, static, simulation), first through the CLI, then through
the Python library, then as structured JSON and TOON outputs. Run it from
the `examples/` directory.

## Setup

Point the imports at the checkout and pick the networks under test.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

NETWORK_INP = PROJECT_ROOT / "tests" / "data" / "valid_network.inp"
NETWORK_JSON = PROJECT_ROOT / "tests" / "data" / "invalid_network.json"

## 01 — The CLI validates in three layers

Static validation as JSON, static validation as TOON, then static plus
simulation validation. The simulation step needs the optional
`simulation` extra.

In [ ]:
def run_cli(*args):
    return subprocess.run(
        [sys.executable, "-m", "epanetparser.core.parse", *args],
        capture_output=True, text=True, cwd=PROJECT_ROOT,
    )

result = run_cli("validate", "-f", str(NETWORK_INP), "--json-output")
payload = json.loads(result.stdout)
print("exit:", result.returncode)
print("is_valid:", payload["is_valid"], "counts:", payload["counts"])

In [ ]:
result = run_cli("validate", "-f", str(NETWORK_INP), "--toon-output")
print("exit:", result.returncode)
print("\n".join(result.stdout.splitlines()[:4]))

## 02 — The same layers through the library

Parse the document, validate it statically, simulate it, then validate
the simulation results against the `hydraulic` dynamic rule set.

In [ ]:
from epanetparser.core.epanettypes.network import WNTREPANETNetwork
from epanetparser.core.validation import (
    ValidationContext,
    Validator,
    validate,
)

network, errors, warnings = WNTREPANETNetwork.from_file(str(NETWORK_INP))
assert network is not None
print(f"parsed {network.name}:", len(network.nodes), "nodes,", len(network.links), "links")

static_report = validate(network)
print("static is_valid:", static_report.is_valid)

In [ ]:
try:
    import wntr
except ImportError:
    wntr = None

if wntr is None:
    print("skipped — install epanetparser[simulation]")
else:
    results = wntr.sim.EpanetSimulator(network.to_wntr()).run_sim()
    validator = Validator.from_context(ValidationContext(custom=["hydraulic"]))
    sim_report = validator.validate_simulation(results, static_report)
    layers = sorted({issue.validation_layer for issue in sim_report})
    print("simulation is_valid:", sim_report.is_valid, "layers:", layers)

## 03 — JSON and TOON describe the same report

One invalid network, two encodings, and a check that they agree on
validity, counts and issue codes.

In [ ]:
bad, errors, _ = WNTREPANETNetwork.from_file(str(NETWORK_JSON))
assert bad is not None
report = validate(bad)

as_json = json.dumps(report.as_dict(), indent=2)
as_toon = report.as_toon()
payload = report.as_dict()
assert ("is_valid: false" in as_toon) == (not payload["is_valid"])
for code in {issue["code"] for issue in payload["issues"]}:
    assert code in as_toon
print(f"agreed on {len(payload['issues'])} issues")
print(f"JSON {len(as_json):,} chars vs TOON {len(as_toon):,} chars")